# Vuln2Action - T5 stage 1 theo paper g?c

Paper g?c d?ng:

- Input: **CVE description** (kh?ng th?m CAPEC v?o input).
- Output: execution flow `Explore / Experiment / Exploit`.
- Split: 80% train / 20% validation.
- T5-base, lr 5e-5, AdamW, cross-entropy.
- Paper ch?y 50 epoch tr?n A100; notebook n?y ch?y **10 epoch tr??c** tr?n
  T4 x2 ?? ki?m tra h?i t?, sau ?? evaluate b?ng ??ng metric paper.

Kh?ng c? exploit source code trong dataset.


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os

hits = sorted(glob.glob("/kaggle/input/**/flow_train.jsonl", recursive=True))
print("flow_train hits:", hits)
if not hits:
    raise FileNotFoundError("flow_train.jsonl not found under /kaggle/input")
FLOW_DIR = Path(hits[0]).parent
os.environ["FLOW_DIR"] = str(FLOW_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("FLOW_DIR:", FLOW_DIR)
for name in ["flow_train.jsonl", "flow_validation.jsonl", "flow_stats.json"]:
    p = FLOW_DIR / name
    print(name, p.exists(), f"{p.stat().st_size / 1024 / 1024:.1f} MiB" if p.exists() else "")


In [ ]:
!python -m torch.distributed.run --nproc_per_node=2 scripts/train_t5.py \
  --dataset "$FLOW_DIR" \
  --output-dir /kaggle/working/t5_stage1_paper \
  --model-name t5-base \
  --target-field target_techniques \
  --max-input-length 256 \
  --max-target-length 256 \
  --per-device-batch-size 8 \
  --gradient-accumulation-steps 4 \
  --learning-rate 5e-5 \
  --num-train-epochs 10 \
  --save-steps 693 \
  --eval-steps 693 \
  --logging-steps 100 \
  --save-total-limit 3 \
  --fp16 \
  --gradient-checkpointing


In [ ]:
from pathlib import Path
import os
base = Path("/kaggle/working/t5_stage1_paper")
final = base / "final"
ckpts = sorted(base.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
print("final exists:", final.exists())
print("checkpoints:", [p.name for p in ckpts][-5:])
CKPT = final if final.exists() else (ckpts[-1] if ckpts else None)
if CKPT is None:
    raise FileNotFoundError("no checkpoint produced")
os.environ["CKPT"] = str(CKPT)
print("CKPT:", CKPT)


In [ ]:
!python scripts/evaluate_t5.py \
  --checkpoint "$CKPT" \
  --dataset "$FLOW_DIR" \
  --split validation \
  --target-field target_techniques \
  --output-dir /kaggle/working/t5_stage1_paper_eval \
  --max-input-length 256 \
  --max-target-length 256 \
  --batch-size 8 \
  --num-beams 4 \
  --fp16


In [ ]:
import json
from pathlib import Path
out = Path("/kaggle/working/t5_stage1_paper_eval")
metrics = json.loads((out / "metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("\n--- sample predictions ---")
for i, line in enumerate((out / "predictions.jsonl").open(encoding="utf-8")):
    if i >= 5:
        break
    row = json.loads(line)
    print(f"\n[{i}] {row['cve_id']}")
    print("REF:", row["reference"][:300])
    print("PRED:", row["prediction"][:300])


## ??c k?t qu?

`bleu` trong `metrics.json` l? **BLEU trung b?nh t?ng c?u, thang 0-1**, gi?ng
Algorithm 2 c?a paper. `bleu_corpus` l? corpus BLEU ?? tham kh?o.

ROUGE c? ?? Precision / Recall / F1 cho ROUGE-1, ROUGE-2, ROUGE-L nh? Table 5.
